In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

def find_repo_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "scripts" / "generate_finer_cam_panderm.py").exists():
            return p.resolve()
    raise FileNotFoundError


REPO = find_repo_root()
for extra in [REPO, REPO / "external" / "PanDerm" / "classification"]:
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from src.thesis.config import *
from src.thesis import data, models, cams, metrics, plotting, audit

EV = data.load_eval()
LESION = data.load_lesions(EV)
QC = data.load_qc()
HUM = data.load_hum(QC, LESION)
X_TEST, Y_MEL, ID_TEST, TEST_DF = data.load_test()
M0, M5 = models.ck5("ha0"), models.ck5("ha5")

AUD = audit.run_audit(EV, LESION, HUM, X_TEST, Y_MEL, ID_TEST, QC, M0, M5)

                                       check  pass         detail
                            index convention  True               
                  test split 987 with 54 MEL  True       987 / 54
                  no annotated image in test  True               
                 lesion masks non degenerate  True             34
             HUM has positives and negatives  True             31
                          ha0 and ha5 differ  True               
                 target follows ground truth  True               
            FinerCAM alpha 0 equals Grad-CAM  True       1.000000
                          AUC rank invariant  True               
                          AUC argument order  True               
              HUM identical to 14d reference  True 31 ref, 31 new
            centre baseline reproduces 0.828  True          0.828
aligned target vs clinician reproduces 0.743  True          0.743

all 13 checks pass
